# MSCOCO_CLIPS1: INT8 Candidate Preflight

Notebook sàng lọc hai cấu hình triển khai: `(1)` encoder và decoder dynamic INT8; `(2)` encoder FP32 kết hợp decoder dynamic INT8. Ứng viên nhỏ nhất vượt toàn bộ gate parity mới được giữ lại.

## Kaggle input bắt buộc — đúng 3 input

1. Output `MOB_MSCOCO_CLIPS1_ONNX_FP32`;
2. output `MOB_MSCOCO_CLIPS1_Calib_Lock`;
3. dataset `ONNX_Runtime_Python313_Offline`.

Runtime: **CPU / Accelerator None / Internet Off**, dự kiến 3–12 phút. Notebook không đọc ảnh Validation và không đọc Test. Calibration lock chỉ được dùng để xác minh authorization cho bước full-quality kế tiếp.

Gate cố định: ONNX reload thành công, output hữu hạn, cosine encoder và decoder tối thiểu 0,995, top-1 agreement tối thiểu 80%, giảm dung lượng ít nhất 10%. Không tự động hạ ngưỡng khi ứng viên thất bại.


In [1]:
import csv
import hashlib
import importlib.util
import json
import math
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

CFG = {'short': 'MSCOCO_CLIPS1', 'teacher': 'CLIP ViT-B/32', 'student': 'MobileNetV3-Large', 'student_id': 'S1', 'dataset': 'MS COCO 2014', 'method': 'Direct Logit-KD', 'seed': 123, 'vocab_name': 'word_vocabulary_minfreq5.json', 'vocab_key': 'id_to_token', 'vocab_size': 10080, 'encoder_key': 'student_encoder_state', 'decoder_key': 'student_decoder_state', 'source_dim': 960, 'encoder_params': 3231024, 'decoder_params': 5759328, 'decoder_batch': 3, 'mask_dtype': 'int64', 'encoder_input': 'image', 'encoder_output': 'visual_memory', 'token_input': 'decoder_input_ids', 'memory_input': 'visual_memory', 'mask_input': 'decoder_attention_mask', 'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225], 'validation_file': 'mscoco_validation.parquet'}
PREFIX = "MOB_MSCOCO_CLIPS1"
EXPERIMENT_ID = PREFIX + "_INT8_Preflight"
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") / EXPERIMENT_ID
PACKAGE_ROOT = WORK_ROOT / "package"
PACKAGE_ROOT.mkdir(parents=True, exist_ok=True)


def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(x) for x in matches])
    return matches[0]


def by_name_and_hash(name, expected_sha256):
    candidates = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    matches = [path for path in candidates if sha256(path) == expected_sha256]
    assert matches, (name, expected_sha256, [str(x) for x in candidates])
    preferred = [path for path in matches if path.parent.name == "package"]
    return (preferred or matches)[0]


fp32_result_path = unique_named(PREFIX + "_ONNX_FP32_COMPLETE.json")
calibration_path = unique_named(PREFIX + "_Calib_Lock.json")
fp32 = read_json(fp32_result_path)
calibration = read_json(calibration_path)

assert fp32["status"] == "complete"
assert fp32["model_id"] == CFG["short"]
assert fp32["selected_seed"] == CFG["seed"]
assert fp32["precision"] == "FP32"
assert fp32["test_set_used"] is False
assert calibration["status"] == "locked"
assert calibration["model_id"] == CFG["short"]
assert calibration["selected_seed"] == CFG["seed"]
assert calibration["calibration_count"] == 128
assert calibration["parity_holdout_count"] == 32
assert calibration["disjoint"] is True
assert calibration["test_set_used"] is False

encoder_input = by_name_and_hash(fp32["encoder"]["file"], fp32["encoder"]["sha256"])
decoder_input = by_name_and_hash(fp32["decoder"]["file"], fp32["decoder"]["sha256"])
encoder_fp32 = WORK_ROOT / fp32["encoder"]["file"]
decoder_fp32 = WORK_ROOT / fp32["decoder"]["file"]
shutil.copy2(encoder_input, encoder_fp32)
shutil.copy2(decoder_input, decoder_fp32)
assert sha256(encoder_fp32) == fp32["encoder"]["sha256"]
assert sha256(decoder_fp32) == fp32["decoder"]["sha256"]

if importlib.util.find_spec("onnxruntime") is None:
    wheels = sorted(INPUT_ROOT.rglob("onnxruntime*.whl"))
    assert wheels, "Hãy gắn dataset ONNX_Runtime_Python313_Offline."
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", str(wheels[0])])
assert importlib.util.find_spec("onnx") is not None
assert importlib.util.find_spec("onnxruntime") is not None

import onnx
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic

encoder_int8 = WORK_ROOT / fp32["encoder"]["file"].replace("_fp32.onnx", "_dynamic_int8.onnx")
decoder_int8 = WORK_ROOT / fp32["decoder"]["file"].replace("_fp32.onnx", "_dynamic_int8.onnx")
quant_args = {"per_channel": True, "reduce_range": False, "weight_type": QuantType.QInt8}
print("[INT8] Quantizing encoder...", flush=True)
quantize_dynamic(model_input=str(encoder_fp32), model_output=str(encoder_int8), **quant_args)
print("[INT8] Quantizing decoder...", flush=True)
quantize_dynamic(model_input=str(decoder_fp32), model_output=str(decoder_int8), **quant_args)
onnx.checker.check_model(onnx.load(str(encoder_int8)))
onnx.checker.check_model(onnx.load(str(decoder_int8)))

providers = ["CPUExecutionProvider"]
sessions = {
    "encoder_fp32": ort.InferenceSession(str(encoder_fp32), providers=providers),
    "encoder_int8": ort.InferenceSession(str(encoder_int8), providers=providers),
    "decoder_fp32": ort.InferenceSession(str(decoder_fp32), providers=providers),
    "decoder_int8": ort.InferenceSession(str(decoder_int8), providers=providers),
}


def cosine(left, right):
    left = np.asarray(left, dtype=np.float64).reshape(-1)
    right = np.asarray(right, dtype=np.float64).reshape(-1)
    denominator = np.linalg.norm(left) * np.linalg.norm(right)
    assert denominator > 0
    return float(np.dot(left, right) / denominator)


def fixed_shape(value_info, fallback):
    result = []
    for index, value in enumerate(value_info.shape):
        if isinstance(value, int) and value > 0:
            result.append(value)
        else:
            result.append(fallback[index])
    return tuple(result)


encoder_session = sessions["encoder_fp32"]
decoder_session = sessions["decoder_fp32"]
encoder_input_meta = encoder_session.get_inputs()[0]
image_shape = fixed_shape(encoder_input_meta, (1, 3, 224, 224))
decoder_inputs = {item.name: item for item in decoder_session.get_inputs()}
token_name = CFG["token_input"]
memory_name = CFG["memory_input"]
mask_name = CFG["mask_input"]
assert {token_name, memory_name, mask_name}.issubset(decoder_inputs)
token_shape = fixed_shape(decoder_inputs[token_name], (CFG["decoder_batch"], 31))
memory_shape = fixed_shape(decoder_inputs[memory_name], (CFG["decoder_batch"], 49, 256))
mask_shape = fixed_shape(decoder_inputs[mask_name], token_shape)
decoder_output_name = decoder_session.get_outputs()[0].name
encoder_output_name = encoder_session.get_outputs()[0].name


def decoder_memory(memory):
    if memory.shape == memory_shape:
        return memory
    assert memory.shape[1:] == memory_shape[1:]
    assert memory.shape[0] == 1 and memory_shape[0] >= 1
    return np.repeat(memory, memory_shape[0], axis=0)


full_rows = []
mixed_rows = []
for case_index, valid_length in enumerate([1, 4, 8, 16, 30]):
    rng = np.random.default_rng(20261102 + case_index)
    image = rng.standard_normal(image_shape, dtype=np.float32)
    tokens = np.zeros(token_shape, dtype=np.int64)
    tokens[:, 0] = 1
    if valid_length > 1:
        tokens[:, 1:valid_length] = rng.integers(
            4, CFG["vocab_size"], size=(token_shape[0], valid_length - 1), dtype=np.int64
        )
    mask_bool = np.zeros(mask_shape, dtype=np.bool_)
    mask_bool[:, :valid_length] = True
    mask = mask_bool.astype(np.int64) if "int64" in decoder_inputs[mask_name].type else mask_bool

    memory_fp32_raw = sessions["encoder_fp32"].run(
        [encoder_output_name], {encoder_input_meta.name: image}
    )[0]
    memory_int8_raw = sessions["encoder_int8"].run(
        [encoder_output_name], {encoder_input_meta.name: image}
    )[0]
    memory_fp32 = decoder_memory(memory_fp32_raw)
    memory_int8 = decoder_memory(memory_int8_raw)
    decoder_base = {token_name: tokens, mask_name: mask}
    logits_fp32 = sessions["decoder_fp32"].run(
        [decoder_output_name], {**decoder_base, memory_name: memory_fp32}
    )[0]
    logits_full = sessions["decoder_int8"].run(
        [decoder_output_name], {**decoder_base, memory_name: memory_int8}
    )[0]
    logits_mixed = sessions["decoder_int8"].run(
        [decoder_output_name], {**decoder_base, memory_name: memory_fp32}
    )[0]
    next_fp32 = logits_fp32[:, valid_length - 1]
    next_full = logits_full[:, valid_length - 1]
    next_mixed = logits_mixed[:, valid_length - 1]
    for array in (memory_fp32_raw, memory_int8_raw, next_fp32, next_full, next_mixed):
        assert np.isfinite(array).all()
    common = {
        "case": case_index,
        "valid_token_length": valid_length,
    }
    full_rows.append({
        **common,
        "encoder_cosine": cosine(memory_fp32_raw, memory_int8_raw),
        "decoder_next_logits_cosine": cosine(next_fp32, next_full),
        "next_token_top1_equal": bool(np.array_equal(next_fp32.argmax(-1), next_full.argmax(-1))),
    })
    mixed_rows.append({
        **common,
        "encoder_cosine": 1.0,
        "decoder_next_logits_cosine": cosine(next_fp32, next_mixed),
        "next_token_top1_equal": bool(np.array_equal(next_fp32.argmax(-1), next_mixed.argmax(-1))),
    })

fp32_bytes = encoder_fp32.stat().st_size + decoder_fp32.stat().st_size


def evaluate(name, rows, encoder_path, decoder_path):
    total_bytes = encoder_path.stat().st_size + decoder_path.stat().st_size
    gates = {
        "encoder_cosine_min": min(row["encoder_cosine"] for row in rows),
        "decoder_cosine_min": min(row["decoder_next_logits_cosine"] for row in rows),
        "top1_agreement": sum(row["next_token_top1_equal"] for row in rows) / len(rows),
        "size_reduction": 1.0 - total_bytes / fp32_bytes,
    }
    passed = (
        gates["encoder_cosine_min"] >= 0.995
        and gates["decoder_cosine_min"] >= 0.995
        and gates["top1_agreement"] >= 0.80
        and gates["size_reduction"] >= 0.10
    )
    return {
        "name": name,
        "status": "passed" if passed else "rejected",
        "encoder_path": encoder_path,
        "decoder_path": decoder_path,
        "total_bytes": total_bytes,
        "total_mib": total_bytes / 1024**2,
        "gates": gates,
        "parity_cases": rows,
    }


candidates = [
    evaluate("full_dynamic_int8", full_rows, encoder_int8, decoder_int8),
    evaluate("fp32_encoder_dynamic_int8_decoder", mixed_rows, encoder_fp32, decoder_int8),
]
passing = [candidate for candidate in candidates if candidate["status"] == "passed"]
assert passing, [{k: v for k, v in item.items() if not k.endswith("_path")} for item in candidates]
selected = min(passing, key=lambda item: item["total_bytes"])

selected_encoder = PACKAGE_ROOT / selected["encoder_path"].name
selected_decoder = PACKAGE_ROOT / selected["decoder_path"].name
vocab_input = by_name_and_hash(fp32["vocabulary"]["file"], fp32["vocabulary"]["sha256"])
vocab_output = PACKAGE_ROOT / fp32["vocabulary"]["file"]
shutil.copy2(selected["encoder_path"], selected_encoder)
shutil.copy2(selected["decoder_path"], selected_decoder)
shutil.copy2(vocab_input, vocab_output)

serializable_candidates = []
for item in candidates:
    serializable_candidates.append({
        key: value for key, value in item.items() if key not in {"encoder_path", "decoder_path"}
    })
result = {
    "schema_version": "mobile_int8_preflight_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "passed",
    "experiment_id": EXPERIMENT_ID,
    "model_id": CFG["short"],
    "teacher": CFG["teacher"],
    "student": CFG["student"],
    "dataset": CFG["dataset"],
    "method": CFG["method"],
    "selected_seed": CFG["seed"],
    "source_fp32_contract_sha256": sha256(fp32_result_path),
    "calibration_lock_sha256": sha256(calibration_path),
    "screened_candidates": serializable_candidates,
    "selection_policy": "Smallest candidate passing all predeclared parity and size gates.",
    "selected_candidate": selected["name"],
    "selected_size": {
        "fp32_reference_bytes": fp32_bytes,
        "selected_bytes": selected["total_bytes"],
        "fp32_reference_mib": fp32_bytes / 1024**2,
        "selected_mib": selected["total_mib"],
        "reduction_fraction": selected["gates"]["size_reduction"],
    },
    "runtime_contract": {
        "encoder_input": encoder_input_meta.name,
        "encoder_output": encoder_output_name,
        "decoder_inputs": [token_name, memory_name, mask_name],
        "decoder_output": decoder_output_name,
        "decoder_batch": token_shape[0],
        "decoder_steps": token_shape[1],
        "mask_dtype": decoder_inputs[mask_name].type,
        "vocabulary_size": CFG["vocab_size"],
    },
    "artifacts": {
        selected_encoder.name: {"bytes": selected_encoder.stat().st_size, "sha256": sha256(selected_encoder)},
        selected_decoder.name: {"bytes": selected_decoder.stat().st_size, "sha256": sha256(selected_decoder)},
        vocab_output.name: {"bytes": vocab_output.stat().st_size, "sha256": sha256(vocab_output)},
    },
    "validation_data_read": False,
    "calibration_data_read": False,
    "test_set_used": False,
    "next_phase_authorized": PREFIX + "_INT8_Full_Validation",
}
result_path = WORK_ROOT / (EXPERIMENT_ID + "_RESULT.json")
write_json(result_path, result)

manifest_rows = []
for artifact in [selected_encoder, selected_decoder, vocab_output, result_path]:
    manifest_rows.append({
        "relative_path": artifact.relative_to(WORK_ROOT).as_posix(),
        "bytes": artifact.stat().st_size,
        "sha256": sha256(artifact),
    })
with (WORK_ROOT / "MANIFEST.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["relative_path", "bytes", "sha256"])
    writer.writeheader()
    writer.writerows(manifest_rows)

print("Status                  : PASSED")
print("Model                   :", CFG["short"])
print("Selected candidate      :", selected["name"])
print("FP32 / selected MiB     :", fp32_bytes / 1024**2, "/", selected["total_mib"])
print("Size reduction          :", selected["gates"]["size_reduction"])
print("Encoder cosine min      :", selected["gates"]["encoder_cosine_min"])
print("Decoder cosine min      :", selected["gates"]["decoder_cosine_min"])
print("Top-1 agreement         :", selected["gates"]["top1_agreement"])
print("Validation / Test read  : False / False")
print("Next                    :", result["next_phase_authorized"])


[INT8] Quantizing encoder...


[INT8] Quantizing decoder...


Status                  : PASSED
Model                   : MSCOCO_CLIPS1
Selected candidate      : fp32_encoder_dynamic_int8_decoder
FP32 / selected MiB     : 44.15689468383789 / 20.497528076171875
Size reduction          : 0.5358023198204133
Encoder cosine min      : 1.0
Decoder cosine min      : 0.9999088792917404
Top-1 agreement         : 1.0
Validation / Test read  : False / False
Next                    : MOB_MSCOCO_CLIPS1_INT8_Full_Validation
